In [ ]:
import csv
import hashlib
import json
import os
import re
import time
import pandas as pd
import requests
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone
import numpy as np

# Raw-data collector v6.1: same transaction / token / balance collection as v5, now driven by
# etherscan_candidates.csv (9 classes, built by build_etherscan_candidates.py). Every contract gets its bytecode,
# creation record and history; verified source is saved as an extra when it exists, so scam contracts that never
# published source are not lost. Wallets are logged and skipped: they have no code, and only some classes hold
# them, so keeping them would let "has bytecode" give the class away. Stops once the class has TARGET_SAMPLES.

# ── CONFIG ────────────────────────────────────────────────────────────────────
API_KEY          = "YOUR_ETHERSCAN_API_KEY"
ETHERSCAN_URL    = "https://api.etherscan.io/v2/api"
CHAIN_ID         = 1
PIPELINE_VERSION = "6.2"

CANDIDATES_CSV = "etherscan_candidates.csv"   # found by name under /kaggle/input or /kaggle/working

# ══════════════════════════════════════════════════════════════════════════════
TARGET_LABEL = 6   # ← ONLY LINE YOU NEED TO CHANGE BETWEEN RUNS
# ══════════════════════════════════════════════════════════════════════════════

DEMO_MAX             = 50     # biggest class in a demo, the others keep their ratio; None for the full run
REQUIRE_SOURCE       = False   # True skips contracts with no verified source on Etherscan
FETCH_IMPLEMENTATION = True    # for proxies, also save the implementation's bytecode and source
TX_LIMIT             = 50000   # same for every class: an address with more than this many records of any type
                               # (normal, internal, ERC-20/721/1155) keeps its oldest records, and all its lists
                               # stop at the same block. Skipping would remove the busiest MEV bots from the class

# ── CLASS MAPPING (matches class_id in etherscan_candidates.csv) ──────────────
CLASS_MAP = {
    0: "benign",
    1: "phishing",
    2: "honeypot",
    3: "rug_pull",
    4: "address_poisoning",
    5: "malicious_mev",
    6: "flash_loan_attack",
    7: "spam_impersonation_token",
    8: "exploit_contract",
}

TARGET_SAMPLES = {                     # contracts to collect per class; stops there
    "benign":                   1900,
    "phishing":                 1850,
    "honeypot":                 1700,
    "rug_pull":                 1870,
    "address_poisoning":        1800,
    "malicious_mev":            1700,
    "flash_loan_attack":        1690,
    "spam_impersonation_token": 1770,
    "exploit_contract":          543,
}
if DEMO_MAX:
    _scale = DEMO_MAX / max(TARGET_SAMPLES.values())
    TARGET_SAMPLES = {name: round(n * _scale) for name, n in TARGET_SAMPLES.items()}
BASE_DIR = Path("/kaggle/working/dataset_demo" if DEMO_MAX else "/kaggle/working/dataset")   # a demo never mixes with real data

# ── AUTO-DERIVED FROM TARGET_LABEL ────────────────────────────────────────────
TARGET_CLASS = CLASS_MAP[TARGET_LABEL]
CAP          = TX_LIMIT
TARGET       = TARGET_SAMPLES[TARGET_CLASS]

print(f"▶ Running label={TARGET_LABEL}  class='{TARGET_CLASS}'  cap={CAP:,}  target={TARGET:,}  require_source={REQUIRE_SOURCE}")

# ── DIRECTORIES ───────────────────────────────────────────────────────────────
class_dir = BASE_DIR / TARGET_CLASS
for sub in [
    class_dir / "transactions",
    class_dir / "token" / "erc-20",
    class_dir / "token" / "erc-721",
    class_dir / "token" / "erc-1155",
    class_dir / "address",
    class_dir / "internal_transactions",
    class_dir / "failed_transactions",
    class_dir / "source",
    class_dir / "bytecode",
]:
    sub.mkdir(exist_ok=True, parents=True)

# ── LABEL-SPECIFIC OUTPUT FILES ───────────────────────────────────────────────
CHECKPOINT_FILE = BASE_DIR / f"checkpoint_label{TARGET_LABEL}.txt"
MASTER_CSV      = BASE_DIR / f"master_dataset_label{TARGET_LABEL}.csv"
FAILED_LOG      = BASE_DIR / f"failed_addresses_label{TARGET_LABEL}.csv"
UNVERIFIED_LOG  = BASE_DIR / f"unverified_addresses_label{TARGET_LABEL}.csv"
WALLET_LOG      = BASE_DIR / f"wallet_addresses_label{TARGET_LABEL}.csv"

MAX_TX_PER_PAGE = 1000   # Etherscan free plan returns at most 1,000 records per call; 10,000 here silently cut every list
SLEEP           = 0.4

# ── INIT LOG FILES ────────────────────────────────────────────────────────────
if not FAILED_LOG.exists():
    with open(FAILED_LOG, 'w', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow(
            ["address", "label", "class_name", "error", "timestamp"]
        )

for _log in (UNVERIFIED_LOG, WALLET_LOG):
    if not _log.exists():
        with open(_log, 'w', newline='') as f:
            csv.writer(f, quoting=csv.QUOTE_ALL).writerow(
                ["address", "label", "class_name", "priority", "timestamp"]
            )

# ── HELPERS ───────────────────────────────────────────────────────────────────
def find(name):
    """A full path is used as given; a bare file name is searched for under /kaggle/input, /kaggle/working, '.'."""
    if Path(name).is_file():
        return Path(name)
    base = Path(name).name
    for top in ("/kaggle/input", "/kaggle/working", "."):
        for root, _, files in os.walk(top):
            if base in files:
                return Path(root) / base
    raise SystemExit(f"cannot find {name}: attach it to the notebook (Add Input)")


def check_daily_limit(msg):
    """Stop the run cleanly on the daily quota; the checkpoint resumes it tomorrow."""
    if "daily" in msg.lower():
        raise SystemExit(f"Etherscan daily limit reached ({msg}). Re-run this cell tomorrow; it resumes.")


def get_class_dirs(label):
    class_name = CLASS_MAP.get(int(label))
    if class_name is None:
        raise ValueError(f"Unknown label '{label}'.")
    cd = BASE_DIR / class_name
    return {
        "class_dir":    cd,
        "trans_dir":    cd / "transactions",
        "internal_dir": cd / "internal_transactions",
        "failed_dir":   cd / "failed_transactions",
        "erc20_dir":    cd / "token" / "erc-20",
        "erc721_dir":   cd / "token" / "erc-721",
        "erc1155_dir":  cd / "token" / "erc-1155",
        "addr_dir":     cd / "address",
        "source_dir":   cd / "source",
        "code_dir":     cd / "bytecode",
        "class_name":   class_name,
    }


def fetch_etherscan(address, action, page=1, offset=MAX_TX_PER_PAGE,
                    startblock=0, endblock=99999999):
    params = {
        "chainid": CHAIN_ID, "module": "account", "action": action,
        "address": address, "startblock": startblock, "endblock": endblock,
        "page": page, "offset": offset, "sort": "asc", "apikey": API_KEY,
    }
    for attempt in range(3):
        try:
            r = requests.get(ETHERSCAN_URL, params=params, timeout=15)
            d = r.json()
            if d['status'] == '1':
                return {"status": "1", "message": d.get("message", "OK"),
                        "result": d['result']}
            check_daily_limit(str(d.get('result', '')))
            if 'rate limit' in str(d.get('result', '')).lower():
                print("    ⏳ Rate limit — sleeping 10 s")
                time.sleep(10)
                continue
            if not d.get('result'):    # the API answered: this address has no records of this type
                return {"status": "0", "message": d.get("message", "No data"),
                        "result": [] if action != "balance" else "0"}
            print(f"    ❌ {action}: {str(d.get('result'))[:120]}")
            time.sleep(2)
        except Exception as e:
            print(f"    ❌ Attempt {attempt + 1} failed: {e}")
            time.sleep(2)
    # never report an API error as "no records": the address is logged as failed and retried on the next run
    raise RuntimeError(f"{action} gave no answer after retries")


def get_source(address):
    """Etherscan getsourcecode record, or None if the API did not answer. SourceCode == '' means unverified."""
    params = {"chainid": CHAIN_ID, "module": "contract", "action": "getsourcecode",
              "address": address, "apikey": API_KEY}
    for attempt in range(3):
        try:
            d = requests.get(ETHERSCAN_URL, params=params, timeout=20).json()
            res = d.get('result')
            if d.get('status') == '1' and isinstance(res, list) and res:
                return res[0]
            check_daily_limit(str(res))
            if 'rate limit' in str(res).lower():
                print("    ⏳ Rate limit — sleeping 10 s")
                time.sleep(10)
                continue
            print(f"    ❌ getsourcecode: {str(res)[:120]}")
            time.sleep(2)
        except Exception as e:
            print(f"    ❌ Attempt {attempt + 1} failed: {e}")
            time.sleep(2)
    return None


def save_source(meta, out_dir, fallback_name):
    """Write every file of a verified contract under out_dir. Returns (n_files, language)."""
    code = meta.get('SourceCode') or ""
    lang = "vyper" if (meta.get('CompilerVersion') or "").lower().startswith("vyper") else "solidity"
    ext  = ".vy" if lang == "vyper" else ".sol"
    files = {}
    if code.startswith("{"):           # multi-file: '{{standard json}}' or '{path: {content}}'
        try:
            spec = json.loads(code[1:-1] if code.startswith("{{") else code)
            srcs = spec.get('sources', spec)
            files = {p: s.get('content', '') for p, s in srcs.items() if isinstance(s, dict)}
        except json.JSONDecodeError:
            files = {}
    if not files:                      # single flattened file
        files = {f"{meta.get('ContractName') or fallback_name}{ext}": code}

    out_dir.mkdir(parents=True, exist_ok=True)
    for path, content in files.items():
        parts = [p for p in PurePosixPath(path.replace("\\", "/")).parts
                 if p not in ("/", "..", ".", "") and not p.endswith(":")]   # stay inside out_dir
        target = out_dir.joinpath(*parts) if parts else out_dir / f"{fallback_name}{ext}"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")

    info = {k: v for k, v in meta.items() if k != 'SourceCode'}
    info['source_files'] = list(files)
    with open(out_dir / "_etherscan_meta.json", 'w') as f:
        json.dump(info, f, indent=2)
    return len(files), lang


def rpc_call(action, **extra):
    """Hex result of an Etherscan JSON-RPC proxy call. An error reply is never mistaken for bytecode."""
    params = {"chainid": CHAIN_ID, "module": "proxy", "action": action, "tag": "latest",
              "apikey": API_KEY, **extra}
    for attempt in range(3):
        try:
            res = requests.get(ETHERSCAN_URL, params=params, timeout=20).json().get('result')
            if isinstance(res, str) and res.startswith('0x'):
                return res
            check_daily_limit(str(res))
            if 'rate limit' in str(res).lower():
                print("    ⏳ Rate limit — sleeping 10 s")
                time.sleep(10)
                continue
            print(f"    ❌ {action}: {str(res)[:120]}")
            time.sleep(2)
        except Exception as e:
            print(f"    ❌ Attempt {attempt + 1} failed: {e}")
            time.sleep(2)
    raise RuntimeError(f"{action} gave no answer after retries")


def get_code(address):
    return bytes.fromhex(rpc_call("eth_getCode", address=address)[2:])


def get_creation(address):
    """Creator, creation tx, timestamp and creation bytecode; None if no contract was ever created at this address."""
    params = {"chainid": CHAIN_ID, "module": "contract", "action": "getcontractcreation",
              "contractaddresses": address, "apikey": API_KEY}
    for attempt in range(3):
        try:
            d = requests.get(ETHERSCAN_URL, params=params, timeout=20).json()
            res = d.get('result')
            if not res:
                return None
            if d.get('status') == '1' and isinstance(res, list):
                return res[0]
            check_daily_limit(str(res))
            if 'rate limit' in str(res).lower():
                print("    ⏳ Rate limit — sleeping 10 s")
                time.sleep(10)
                continue
            print(f"    ❌ getcontractcreation: {str(res)[:120]}")
            time.sleep(2)
        except Exception as e:
            print(f"    ❌ Attempt {attempt + 1} failed: {e}")
            time.sleep(2)
    raise RuntimeError("getcontractcreation gave no answer after retries")


def strip_metadata(code):
    """Remove the CBOR metadata at the end: it holds the compiler version and a source hash."""
    if len(code) > 2:
        n = int.from_bytes(code[-2:], "big")
        if 0 < n <= len(code) - 2 and code[-n - 2] in (0xa1, 0xa2, 0xa3):
            return code[:-n - 2]
    return code


def opcodes(code):
    """Opcode sequence with PUSH data removed."""
    ops, i = [], 0
    while i < len(code):
        op = code[i]
        if 0x60 <= op <= 0x7f:         # PUSH1..PUSH32 carry 1..32 bytes of data
            ops.append("PUSH")
            i += op - 0x5f + 1
        else:
            ops.append(f"{op:02x}")
            i += 1
    return ops


def code_group(code):
    """Same hash = same contract apart from constants: keep each group on one side of the train/test split."""
    return hashlib.sha256(" ".join(opcodes(strip_metadata(code))).encode()).hexdigest() if code else None


EIP1167 = re.compile(r"363d3d373d3d3d363d73([0-9a-f]{40})5af43d82803e903d91602b57fd5bf3")
EIP1967_SLOT = "0x360894a13ba1a3210667c828492db98dca3e2076cc3735a920a3ca505d382bbc"


def find_implementation(address, code, src):
    """(implementation, how it was found) for a proxy, else (None, None). Beacon proxies are not resolved."""
    if src.get('Proxy') == '1' and src.get('Implementation'):
        return src['Implementation'].lower(), "etherscan"
    m = EIP1167.fullmatch(code.hex())
    if m:
        return "0x" + m.group(1), "eip1167"
    if "f4" in opcodes(strip_metadata(code)):      # DELEGATECALL; other contracts cannot be EIP-1967 proxies
        slot = int(rpc_call("eth_getStorageAt", address=address, position=EIP1967_SLOT)[2:] or "0", 16)
        if slot:
            return f"0x{slot & (2**160 - 1):040x}", "eip1967"
    return None, None


def fetch_all_paginated(address, action, cap: int, endblock=99999999):
    all_results = []
    startblock  = 0
    truncated   = False

    while True:
        data = fetch_etherscan(address, action,
                               page=1, offset=MAX_TX_PER_PAGE,
                               startblock=startblock, endblock=endblock)

        if data['status'] == '0' or not data['result']:
            break

        batch     = data['result']
        full_page = len(batch) >= MAX_TX_PER_PAGE
        whole     = batch
        if full_page:
            # a full page can end in the middle of a block: drop that block's rows here and
            # fetch the block again from its start, so none of its transactions are skipped
            last_block = int(batch[-1]['blockNumber'])
            whole = [tx for tx in batch if int(tx['blockNumber']) < last_block]
            batch = whole or batch
        remaining = cap - len(all_results)

        if len(batch) > remaining or (len(batch) == remaining and full_page):
            all_results.extend(batch[:remaining])
            truncated = True
            print(f"    ⚠️  Limit of {cap:,} records reached [{action}] — keeping the oldest")
            break

        all_results.extend(batch)

        if not full_page:
            break
        if not whole:                  # one block alone fills a page; Etherscan cannot page inside it
            truncated = True
            break
        if last_block >= endblock:
            break

        startblock = last_block
        print(f"    ↳ {action}: next startblock={startblock} "
              f"(total so far: {len(all_results):,})")
        time.sleep(0.2)

    return {
        "status":        "1" if all_results else "0",
        "message":       "OK" if all_results else "No data found",
        "result":        all_results,
        "truncated":     truncated,
        "total_fetched": len(all_results),
    }


def save_checkpoint(address, stats_row):
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(f"{address}\n")
    df_row = pd.DataFrame([stats_row])
    if not MASTER_CSV.exists():
        df_row.to_csv(MASTER_CSV, index=False)
    else:
        df_row.to_csv(MASTER_CSV, mode='a', header=False, index=False)


def log_failed(address, label, class_name, error):
    with open(FAILED_LOG, 'a', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow([
            address, label, class_name,
            str(error), datetime.now(timezone.utc).isoformat()
        ])


def log_unverified(address, label, class_name, priority):
    with open(UNVERIFIED_LOG, 'a', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow([
            address, label, class_name, priority,
            datetime.now(timezone.utc).isoformat()
        ])
    with open(CHECKPOINT_FILE, 'a') as f:     # never asked again
        f.write(f"{address}\n")


def log_wallet(address, label, class_name, priority):
    with open(WALLET_LOG, 'a', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow([
            address, label, class_name, priority,
            datetime.now(timezone.utc).isoformat()
        ])
    with open(CHECKPOINT_FILE, 'a') as f:     # never asked again
        f.write(f"{address}\n")


# ── LOAD + FILTER TO TARGET LABEL ONLY (in priority order) ───────────────────
cand_path = find(CANDIDATES_CSV)
api_df = pd.read_csv(cand_path)
api_df = (api_df[api_df['class_id'] == TARGET_LABEL]
          .sort_values('priority').reset_index(drop=True))
print(f"Candidates for label={TARGET_LABEL} ({TARGET_CLASS}): {len(api_df):,}  from {cand_path}")

# ── RESUME FROM CHECKPOINT ────────────────────────────────────────────────────
collected = len(pd.read_csv(MASTER_CSV)) if MASTER_CSV.exists() else 0
if CHECKPOINT_FILE.exists():
    completed = set(CHECKPOINT_FILE.read_text().strip().split('\n'))
    completed.discard('')
    api_df = api_df[~api_df['address'].isin(completed)].reset_index(drop=True)
    print(f"Already checked: {len(completed):,}  |  Collected: {collected:,}  |  "
          f"Remaining candidates: {len(api_df):,}")

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
failed_count_session     = 0
unverified_count_session = 0
wallet_count_session     = 0

for idx, row in api_df.iterrows():
    if collected >= TARGET:
        print(f"\n🎯 Target reached: {collected:,} contracts for {TARGET_CLASS}")
        break

    addr  = row['address']
    label = TARGET_LABEL

    print(f"\n[{idx + 1}/{len(api_df)}] {addr}  (label={label} / {TARGET_CLASS})  "
          f"collected {collected:,}/{TARGET:,}  priority={row['priority']}")

    try:
        dirs = get_class_dirs(label)

        collection_timestamp = datetime.now(timezone.utc).isoformat()
        now_ts               = datetime.now(timezone.utc).timestamp()

        # ── Verified source, if any (1 call) ──────────────────────────────────
        src = get_source(addr)
        if src is None:
            raise RuntimeError("getsourcecode gave no answer")   # not checkpointed, retried next run
        has_source = bool(src.get('SourceCode'))
        if not has_source and REQUIRE_SOURCE:
            print("  ⛔ No verified source — skipped")
            log_unverified(addr, label, TARGET_CLASS, row['priority'])
            unverified_count_session += 1
            time.sleep(0.2)
            continue

        # ── Bytecode and creation record; a self-destructed contract has only the second ──
        code     = get_code(addr)
        creation = get_creation(addr)
        if not code and creation is None:
            print("  👛 Wallet, no contract code — skipped")
            log_wallet(addr, label, TARGET_CLASS, row['priority'])
            wallet_count_session += 1
            time.sleep(0.2)
            continue
        code_status = "live" if code else "destroyed"

        # ── Fetch all data, up to TX_LIMIT per type; every list ends at the same block ──
        fetched, cut_block = {}, None
        for action in ("txlist", "txlistinternal", "tokentx", "tokennfttx", "token1155tx"):
            fetched[action] = fetch_all_paginated(addr, action, cap=TX_LIMIT, endblock=cut_block or 99999999)
            if fetched[action]['truncated']:
                cut_block = int(fetched[action]['result'][-1]['blockNumber'])
        if cut_block:                  # the last block may be incomplete, so it is left out of every list
            for data in fetched.values():
                data['result']        = [tx for tx in data['result'] if int(tx['blockNumber']) < cut_block]
                data['status']        = "1" if data['result'] else "0"
                data['total_fetched'] = len(data['result'])

        normal_tx   = fetched["txlist"]
        internal_tx = fetched["txlistinternal"]
        erc20_tx    = fetched["tokentx"]
        erc721_tx   = fetched["tokennfttx"]
        erc1155_tx  = fetched["token1155tx"]
        balance     = fetch_etherscan(addr, "balance", 1, 1)

        n_source_files, source_language = 0, None
        impl_addr, impl_found_by = find_implementation(addr, code, src) if code else (None, None)
        impl_code, impl_files = b"", 0
        if FETCH_IMPLEMENTATION and impl_addr:
            impl_code = get_code(impl_addr)
            impl = get_source(impl_addr)
            if impl and impl.get('SourceCode'):
                impl_files, _ = save_source(impl, dirs['source_dir'] / addr / "implementation", impl_addr)
        if has_source:
            n_source_files, source_language = save_source(src, dirs['source_dir'] / addr, addr)

        creation_code = bytes.fromhex(((creation or {}).get('creationBytecode') or '0x')[2:])
        group = code_group(impl_code or code or creation_code)   # all proxies look alike, so group by implementation
        with open(dirs['code_dir'] / f"{addr}.json", 'w') as f:
            json.dump({
                "address":                 addr,
                "code_status":             code_status,
                "bytecode":                "0x" + code.hex(),
                "code_group":              group,
                "implementation":          impl_addr,
                "implementation_found_by": impl_found_by,
                "implementation_bytecode": "0x" + impl_code.hex(),
                "creation":                creation,
            }, f, indent=2)
        print(f"  🔢 {code_status} bytecode {len(code):,} bytes"
              + (f"  proxy[{impl_found_by}] -> {impl_addr}" if impl_addr else "")
              + (f"  📄 {src.get('ContractName')} {src.get('CompilerVersion')} {n_source_files} file(s)" if has_source
                 else "  no verified source")
              + (f"  + implementation source {impl_files}" if impl_files else ""))

        raw_bal     = balance.get('result', '0')
        balance_wei = int(raw_bal) if str(raw_bal).isdigit() else 0

        successful_tx        = [tx for tx in normal_tx['result'] if tx.get('isError') == '0']
        failed_tx            = [tx for tx in normal_tx['result'] if tx.get('isError') == '1']
        contract_creation_tx = [tx for tx in normal_tx['result'] if tx.get('contractAddress')]
        approval_tx          = [tx for tx in normal_tx['result']
                                if tx.get('input', '')[:10] == '0x095ea7b3']

        # ── Stats dict ────────────────────────────────────────────────────────
        stats = {
            'address':             addr,
            'label':               label,
            'class_name':          TARGET_CLASS,
            'candidate_year':      row.get('year'),
            'candidate_evidence':  row.get('evidence'),
            'candidate_priority':  row.get('priority'),
            'collected_at_utc':    collection_timestamp,
            'pipeline_version':    PIPELINE_VERSION,
            'has_source':          has_source,      # never use as a model feature: scams rarely publish source
            'contract_name':       src.get('ContractName'),
            'compiler_version':    src.get('CompilerVersion'),
            'source_language':     source_language,
            'n_source_files':      n_source_files,
            'is_proxy':            bool(impl_addr),
            'implementation':      impl_addr,
            'implementation_found_by': impl_found_by,
            'implementation_files': impl_files,
            'code_status':         code_status,
            'bytecode_size':       len(code),
            'code_group':          group,
            'creator':             (creation or {}).get('contractCreator'),
            'creation_tx':         (creation or {}).get('txHash'),
            'creation_timestamp':  (creation or {}).get('timestamp'),
            'balance_wei':         balance_wei,
            'balance_eth':         balance_wei / 1e18,
            'total_tx_count':      len(normal_tx['result']),
            'successful_tx_count': len(successful_tx),
            'failed_tx_count':     len(failed_tx),
            'internal_tx_count':   len(internal_tx['result']),
            'erc20_tx_count':      len(erc20_tx['result']),
            'erc721_tx_count':     len(erc721_tx['result']),
            'erc1155_tx_count':    len(erc1155_tx['result']),
            'contract_creation_count': len(contract_creation_tx),
            'approval_count':      len(approval_tx),
            'cap_applied':         CAP,
            'data_is_partial':     bool(cut_block),  # never use as a model feature: it follows from the limit, not behaviour
            'window_end_block':    cut_block,        # records stop before this block; empty if the history is complete
            'normal_tx_truncated':   normal_tx['truncated'],
            'internal_tx_truncated': internal_tx['truncated'],
            'erc20_tx_truncated':    erc20_tx['truncated'],
            # Working sets (removed before save)
            'unique_receivers':          set(),
            'unique_senders':            set(),
            'total_value_sent_wei':      0,
            'total_value_received_wei':  0,
            'total_gas_used':            0,
            'avg_gas_price':             0,
            'first_tx_timestamp':        None,
            'last_tx_timestamp':         None,
            'unique_erc20_tokens':       set(),
            'unique_erc721_tokens':      set(),
            'unique_erc1155_tokens':     set(),
            'method_ids':                Counter(),
            'unique_method_ids':         set(),
            'unique_approved_contracts': set(),
        }

        gas_prices        = []
        tx_timestamps     = []
        zero_value_count  = 0
        first_approval_ts = None
        last_approval_ts  = None

        for tx in normal_tx['result']:
            tx_from = (tx.get('from') or '').lower()
            tx_to   = (tx.get('to')   or '').lower()
            value   = int(tx.get('value', 0))

            if tx_from == addr.lower():
                if tx_to:
                    stats['unique_receivers'].add(tx_to)
                stats['total_value_sent_wei'] += value
            else:
                if tx_from:
                    stats['unique_senders'].add(tx_from)
                stats['total_value_received_wei'] += value

            if value == 0:
                zero_value_count += 1

            stats['total_gas_used'] += int(tx.get('gasUsed', 0))
            if tx.get('gasPrice'):
                gas_prices.append(int(tx['gasPrice']))

            input_data = tx.get('input', '0x')
            if len(input_data) >= 10:
                method_id = input_data[:10]
                stats['method_ids'][method_id] += 1
                stats['unique_method_ids'].add(method_id)
                if method_id == '0x095ea7b3' and tx_to:
                    stats['unique_approved_contracts'].add(tx_to)

            timestamp = int(tx.get('timeStamp', 0))
            tx_timestamps.append(timestamp)

            if stats['first_tx_timestamp'] is None:
                stats['first_tx_timestamp'] = timestamp
            stats['last_tx_timestamp'] = timestamp

            if input_data[:10] == '0x095ea7b3':
                if first_approval_ts is None:
                    first_approval_ts = timestamp
                last_approval_ts = timestamp

        stats['avg_gas_price']       = sum(gas_prices) / len(gas_prices) if gas_prices else 0
        stats['zero_value_tx_count'] = zero_value_count

        if len(tx_timestamps) >= 2:
            deltas = [tx_timestamps[i+1] - tx_timestamps[i]
                      for i in range(len(tx_timestamps) - 1)]
            stats['avg_time_between_tx_sec'] = float(np.mean(deltas))
            stats['std_time_between_tx_sec'] = float(np.std(deltas))
        else:
            stats['avg_time_between_tx_sec'] = None
            stats['std_time_between_tx_sec'] = None

        stats['first_approval_timestamp'] = first_approval_ts
        stats['last_approval_timestamp']  = last_approval_ts

        for tx in erc20_tx['result']:
            ca = (tx.get('contractAddress') or '').lower()
            if ca:
                stats['unique_erc20_tokens'].add(ca)

        for tx in erc721_tx['result']:
            ca = (tx.get('contractAddress') or '').lower()
            if ca:
                stats['unique_erc721_tokens'].add(ca)

        for tx in erc1155_tx['result']:
            ca = (tx.get('contractAddress') or '').lower()
            if ca:
                stats['unique_erc1155_tokens'].add(ca)

        stats['first_tx_datetime'] = (
            pd.to_datetime(stats['first_tx_timestamp'], unit='s')
            if stats['first_tx_timestamp'] is not None else None
        )
        stats['last_tx_datetime'] = (
            pd.to_datetime(stats['last_tx_timestamp'], unit='s')
            if stats['last_tx_timestamp'] is not None else None
        )
        stats['active_span_days'] = (
            (stats['last_tx_timestamp'] - stats['first_tx_timestamp']) / 86400
            if (stats['first_tx_timestamp'] and stats['last_tx_timestamp'])
            else 0
        )
        stats['account_age_days'] = (
            (now_ts - stats['first_tx_timestamp']) / 86400
            if stats['first_tx_timestamp'] is not None else 0
        )

        # ── Scalar counts ─────────────────────────────────────────────────────
        stats['unique_receivers_count']          = len(stats['unique_receivers'])
        stats['unique_senders_count']            = len(stats['unique_senders'])
        stats['unique_erc20_tokens_count']       = len(stats['unique_erc20_tokens'])
        stats['unique_erc721_tokens_count']      = len(stats['unique_erc721_tokens'])
        stats['unique_erc1155_tokens_count']     = len(stats['unique_erc1155_tokens'])
        stats['unique_method_ids_count']         = len(stats['unique_method_ids'])
        stats['unique_approved_contracts_count'] = len(stats['unique_approved_contracts'])

        if stats['method_ids']:
            stats['most_common_method_id']    = stats['method_ids'].most_common(1)[0][0]
            stats['most_common_method_count'] = stats['method_ids'].most_common(1)[0][1]
        else:
            stats['most_common_method_id']    = None
            stats['most_common_method_count'] = 0

        total = stats['total_tx_count']
        stats['failed_tx_ratio'] = stats['failed_tx_count'] / total if total else 0
        stats['sent_to_received_value_ratio'] = (
            stats['total_value_sent_wei'] / stats['total_value_received_wei']
            if stats['total_value_received_wei'] else None
        )
        outgoing = sum(
            1 for tx in normal_tx['result']
            if (tx.get('from') or '').lower() == addr.lower()
        )
        stats['fan_out_ratio'] = (
            stats['unique_receivers_count'] / outgoing if outgoing else 0
        )

        # ── ETH convenience columns ───────────────────────────────────────────
        stats['total_value_sent_eth']     = stats['total_value_sent_wei']     / 1e18
        stats['total_value_received_eth'] = stats['total_value_received_wei'] / 1e18
        stats['avg_gas_price_gwei']       = stats['avg_gas_price'] / 1e9

        # ── Remove working sets before CSV write ──────────────────────────────
        for key in ['unique_receivers', 'unique_senders', 'unique_erc20_tokens',
                    'unique_erc721_tokens', 'unique_erc1155_tokens',
                    'method_ids', 'unique_method_ids', 'unique_approved_contracts']:
            del stats[key]

        # ── Print summary ─────────────────────────────────────────────────────
        partial_flag = f"⚠️  OLDEST RECORDS ONLY, before block {cut_block}" if cut_block else "✅ FULL"
        print(f"  txs={stats['total_tx_count']:,}  bal={stats['balance_eth']:.4f} ETH  {partial_flag}")

        # ── Save JSON files ───────────────────────────────────────────────────
        if normal_tx['result']:
            with open(dirs['trans_dir'] / f"{addr}.json", 'w') as f:
                json.dump(normal_tx, f, indent=2)

        if internal_tx['result']:
            with open(dirs['internal_dir'] / f"{addr}.json", 'w') as f:
                json.dump(internal_tx, f, indent=2)

        if failed_tx:
            with open(dirs['failed_dir'] / f"{addr}.json", 'w') as f:
                json.dump({"status": "1", "result": failed_tx}, f, indent=2)

        for dir_key, data in [('erc20_dir',   erc20_tx),
                               ('erc721_dir',  erc721_tx),
                               ('erc1155_dir', erc1155_tx)]:
            with open(dirs[dir_key] / f"{addr}.json", 'w') as f:
                json.dump({
                    "status":    "1" if data['result'] else "0",
                    "message":   "OK" if data['result'] else "No token found",
                    "result":    data['result'],
                    "truncated": data['truncated'],
                }, f, indent=2)

        with open(dirs['addr_dir'] / f"{addr}.json", 'w') as f:
            json.dump({
                "collected_at_utc":    collection_timestamp,
                "pipeline_version":    PIPELINE_VERSION,
                "class_name":          TARGET_CLASS,
                "cap_applied":         CAP,
                "data_is_partial":     stats['data_is_partial'],
                "has_source":          has_source,
                "contract_name":       src.get('ContractName'),
                "compiler_version":    src.get('CompilerVersion'),
                "is_proxy":            bool(impl_addr),
                "implementation":      impl_addr,
                "code_status":         code_status,
                "truncation_flags": {
                    "normal_tx":   normal_tx['truncated'],
                    "internal_tx": internal_tx['truncated'],
                    "erc20_tx":    erc20_tx['truncated'],
                    "erc721_tx":   erc721_tx['truncated'],
                    "erc1155_tx":  erc1155_tx['truncated'],
                },
                "statuses": {
                    "normal_tx":   normal_tx['status'],
                    "internal_tx": internal_tx['status'],
                    "erc20_tx":    erc20_tx['status'],
                    "erc721_tx":   erc721_tx['status'],
                    "erc1155_tx":  erc1155_tx['status'],
                    "balance":     balance['status'],
                },
                "balance_wei":             balance_wei,
                "total_tx_count":          stats['total_tx_count'],
                "internal_tx_count":       stats['internal_tx_count'],
                "erc20_tx_count":          stats['erc20_tx_count'],
                "erc721_tx_count":         stats['erc721_tx_count'],
                "erc1155_tx_count":        stats['erc1155_tx_count'],
                "failed_tx_count":         stats['failed_tx_count'],
                "contract_creation_count": stats['contract_creation_count'],
                "approval_count":          stats['approval_count'],
            }, f, indent=2)

        save_checkpoint(addr, stats)
        collected += 1
        print("  ✅ Checkpointed")

    except Exception as e:
        print(f"  ❌ FAILED: {e}")
        log_failed(addr, label, TARGET_CLASS, e)
        failed_count_session += 1
        continue

    time.sleep(SLEEP)

# ── SESSION SUMMARY ───────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  Run complete  [label={TARGET_LABEL} / {TARGET_CLASS}]")
print(f"  Collected         : {collected:,} / {TARGET:,}"
      + ("" if collected >= TARGET else "  (candidate list exhausted or run stopped early)"))
if MASTER_CSV.exists():
    _m = pd.read_csv(MASTER_CSV)
    print(f"  With verified source: {int(_m['has_source'].sum()):,}  |  self-destructed: "
          f"{int((_m['code_status'] == 'destroyed').sum()):,}  |  proxies: {int(_m['is_proxy'].sum()):,}  |  "
          f"distinct code groups: {_m['code_group'].nunique():,}")
    print(f"  Cut at {TX_LIMIT:,} records: {int(_m['data_is_partial'].sum()):,}  (report this count per class in the paper)")
print(f"  Unverified skipped: {unverified_count_session:,}")
print(f"  Wallets skipped   : {wallet_count_session:,}")
print(f"  Failed            : {failed_count_session:,}")
print(f"  Master dataset    : {MASTER_CSV}")
print(f"  Source folder     : {class_dir / 'source'}")
print(f"  Bytecode folder   : {class_dir / 'bytecode'}")
print(f"  Unverified log    : {UNVERIFIED_LOG}")
print(f"  Wallet log        : {WALLET_LOG}")
print(f"  Failed log        : {FAILED_LOG}")
print(f"{'═'*60}")
